In [ ]:
"""
Stage 2 — Feature Engineering
Builds all input features for the LSTM stock return model.
Reads CSVs from data/raw/, writes enriched CSVs to data/features/.

Feature groups:
  A. Return features       — what the stock actually did
  B. Momentum indicators   — RSI, MACD (is it trending?)
  C. Volatility indicators — Bollinger Bands, ATR (how much is it moving?)
  D. Volume signals        — z-score, OBV (is smart money moving?)
  E. Market context        — beta-adjusted return vs Nifty50 (stock vs market)
  F. Target variable       — next-day log return (what we want to predict)

CRITICAL — Lookahead bias rule:
  Every feature at row t uses ONLY data from rows 0..t.
  Rolling windows, shifts, and indicator lookback periods all respect this.
  This is the single most common mistake in quant fresher projects —
  make sure you can explain this in an interview.
"""

import pandas as pd
import numpy as np
import os

RAW_DIR      = "data/raw"
FEATURE_DIR  = "data/features"
INDEX_FILE   = "data/raw/NSEI.csv"   # Nifty50 index, saved by Stage 1


# ═══════════════════════════════════════════════════════════════════════════════
# A. RETURN FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

def add_return_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Log returns are preferred over simple percentage returns in quant finance
    because they're additive over time and more normally distributed —
    both useful properties for neural network inputs.

    log_return_1d  = log(Close_t / Close_{t-1})  ← most important feature
    log_return_5d  = log(Close_t / Close_{t-5})  ← weekly momentum
    log_return_10d = log(Close_t / Close_{t-10}) ← two-week momentum
    log_return_21d = log(Close_t / Close_{t-21}) ← monthly momentum
    """
    close = df["Close"]

    df["log_return_1d"]  = np.log(close / close.shift(1))
    df["log_return_5d"]  = np.log(close / close.shift(5))
    df["log_return_10d"] = np.log(close / close.shift(10))
    df["log_return_21d"] = np.log(close / close.shift(21))

    return df


# ═══════════════════════════════════════════════════════════════════════════════
# B. MOMENTUM INDICATORS
# ═══════════════════════════════════════════════════════════════════════════════

def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """
    RSI (Relative Strength Index) — measures speed and magnitude of price moves.
    Range: 0–100. Above 70 = overbought, below 30 = oversold.
    We return the raw RSI value (not a signal) so the LSTM can learn
    its own thresholds from data rather than using our hardcoded ones.

    We implement manually here to avoid a TA-Lib dependency.
    """
    delta  = series.diff()
    gain   = delta.clip(lower=0)
    loss   = -delta.clip(upper=0)

    # Wilder's smoothing (equivalent to EMA with alpha = 1/period)
    avg_gain = gain.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, min_periods=period, adjust=False).mean()

    rs  = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))
    return rsi


def compute_macd(series: pd.Series,
                 fast: int = 12, slow: int = 26, signal: int = 9):
    """
    MACD (Moving Average Convergence Divergence).
    Returns three series:
      macd_line      = EMA(fast) - EMA(slow)
      macd_signal    = EMA(macd_line, signal)
      macd_histogram = macd_line - macd_signal (crossover signal)

    We feed all three to the LSTM separately so it can learn
    which component of MACD is predictive for each stock.
    """
    ema_fast   = series.ewm(span=fast,   adjust=False).mean()
    ema_slow   = series.ewm(span=slow,   adjust=False).mean()
    macd_line  = ema_fast - ema_slow
    macd_sig   = macd_line.ewm(span=signal, adjust=False).mean()
    macd_hist  = macd_line - macd_sig
    return macd_line, macd_sig, macd_hist


def add_momentum_features(df: pd.DataFrame) -> pd.DataFrame:
    df["rsi_14"]           = compute_rsi(df["Close"], period=14)

    macd_line, macd_sig, macd_hist = compute_macd(df["Close"])
    df["macd_line"]        = macd_line
    df["macd_signal"]      = macd_sig
    df["macd_histogram"]   = macd_hist

    # Price relative to its own moving averages — captures trend state
    df["close_to_sma20"]   = df["Close"] / df["Close"].rolling(20).mean() - 1
    df["close_to_sma50"]   = df["Close"] / df["Close"].rolling(50).mean() - 1

    # Rate of change — normalized momentum over N days
    df["roc_5"]            = df["Close"].pct_change(5)
    df["roc_21"]           = df["Close"].pct_change(21)

    return df


# ═══════════════════════════════════════════════════════════════════════════════
# C. VOLATILITY INDICATORS
# ═══════════════════════════════════════════════════════════════════════════════

def add_volatility_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Bollinger Bands — price relative to its own volatility envelope.
    bb_width    = how wide the bands are (realized vol proxy)
    bb_position = where price sits within the band (-1 to +1 roughly)

    ATR (Average True Range) — raw volatility in price terms.
    Normalised by Close so it's comparable across different price levels.

    Realised vol — rolling std of log returns, annualised.
    A direct measure of how much the stock is moving day-to-day.
    """
    close  = df["Close"]
    high   = df["High"]
    low    = df["Low"]

    # Bollinger Bands (20-day, 2 std)
    sma20       = close.rolling(20).mean()
    std20       = close.rolling(20).std()
    bb_upper    = sma20 + 2 * std20
    bb_lower    = sma20 - 2 * std20

    df["bb_width"]    = (bb_upper - bb_lower) / sma20          # normalised width
    df["bb_position"] = (close - bb_lower) / (bb_upper - bb_lower)  # 0=lower band, 1=upper

    # ATR
    prev_close  = close.shift(1)
    true_range  = pd.concat([
        high - low,
        (high - prev_close).abs(),
        (low  - prev_close).abs()
    ], axis=1).max(axis=1)
    df["atr_14_norm"] = true_range.ewm(span=14, adjust=False).mean() / close

    # Realised volatility (annualised)
    log_ret     = np.log(close / close.shift(1))
    df["realised_vol_21"] = log_ret.rolling(21).std() * np.sqrt(252)

    return df


# ═══════════════════════════════════════════════════════════════════════════════
# D. VOLUME SIGNALS
# ═══════════════════════════════════════════════════════════════════════════════

def add_volume_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Volume z-score: how unusual is today's volume vs the past 20 days?
    Values above +2 or below -2 signal abnormal activity.
    This is a simple proxy for institutional flow / news events.

    OBV (On-Balance Volume): cumulates volume with sign based on price direction.
    Captures whether volume is accumulating (bullish) or distributing (bearish).
    We normalise OBV by its own rolling mean so it's stationary enough for LSTM.
    """
    volume = df["Volume"]
    close  = df["Close"]

    # Volume z-score (rolling 20-day)
    vol_mean = volume.rolling(20).mean()
    vol_std  = volume.rolling(20).std()
    df["volume_zscore"] = (volume - vol_mean) / vol_std.replace(0, np.nan)

    # Volume ratio — today vs 5-day average (simpler, more stable)
    df["volume_ratio_5d"] = volume / volume.rolling(5).mean()

    # OBV (normalised)
    direction = np.sign(close.diff())
    obv       = (volume * direction).cumsum()
    obv_mean  = obv.rolling(20).mean()
    obv_std   = obv.rolling(20).std()
    df["obv_zscore"] = (obv - obv_mean) / obv_std.replace(0, np.nan)

    return df


# ═══════════════════════════════════════════════════════════════════════════════
# E. MARKET CONTEXT (stock vs Nifty50)
# ═══════════════════════════════════════════════════════════════════════════════

def add_market_features(df: pd.DataFrame,
                        index_df: pd.DataFrame) -> pd.DataFrame:
    """
    Beta-adjusted relative return:
      = stock's log return - (rolling_beta × index log return)

    This tells you how much the stock moved BEYOND what the market explains.
    Positive = stock is outperforming the market today (idiosyncratic strength).
    Negative = stock is underperforming even accounting for market movement.

    Rolling beta (60-day) is recalculated every day using only past data —
    no lookahead bias.
    """
    index_ret = np.log(index_df["Close"] / index_df["Close"].shift(1))
    index_ret.name = "index_ret"

    stock_ret = df["log_return_1d"]

    # Align on date index
    aligned = pd.concat([stock_ret, index_ret], axis=1).dropna()

    # Rolling 60-day beta = cov(stock, index) / var(index)
    rolling_cov  = aligned["log_return_1d"].rolling(60).cov(aligned["index_ret"])
    rolling_var  = aligned["index_ret"].rolling(60).var()
    rolling_beta = rolling_cov / rolling_var.replace(0, np.nan)

    df["rolling_beta"]    = rolling_beta
    df["index_ret"]       = index_ret
    df["relative_return"] = stock_ret - (rolling_beta * index_ret)

    return df


# ═══════════════════════════════════════════════════════════════════════════════
# F. TARGET VARIABLE
# ═══════════════════════════════════════════════════════════════════════════════

def add_target(df: pd.DataFrame,
               horizon: int = 1) -> pd.DataFrame:
    """
    Target = log return N days ahead.
    We use shift(-horizon) to peek forward — this is intentional ONLY for the
    target column. All feature columns must never use future data.

    For classification (direction): target_direction = 1 if return > 0 else 0
    We keep both — you can switch loss functions to test regression vs clf.
    """
    df[f"target_return_{horizon}d"]    = df["log_return_1d"].shift(-horizon)
    df[f"target_direction_{horizon}d"] = (df[f"target_return_{horizon}d"] > 0).astype(int)
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# ROLLING Z-SCORE NORMALISATION (avoids lookahead bias vs global min-max)
# ═══════════════════════════════════════════════════════════════════════════════

FEATURE_COLS = [
    "log_return_1d", "log_return_5d", "log_return_10d", "log_return_21d",
    "rsi_14",
    "macd_line", "macd_signal", "macd_histogram",
    "close_to_sma20", "close_to_sma50",
    "roc_5", "roc_21",
    "bb_width", "bb_position",
    "atr_14_norm", "realised_vol_21",
    "volume_zscore", "volume_ratio_5d", "obv_zscore",
    "rolling_beta", "index_ret", "relative_return",
]

def rolling_zscore_normalise(df: pd.DataFrame,
                              cols: list[str],
                              window: int = 252) -> pd.DataFrame:
    """
    Normalise each feature using its own rolling 252-day (1yr) mean and std.
    This is the correct approach for time-series ML:
      - Global normalisation uses future data to compute mean/std → LOOKAHEAD BIAS
      - Rolling normalisation only uses past data → SAFE

    Values more than 3 std from the mean are clipped to prevent extreme
    values destabilising LSTM training.
    """
    for col in cols:
        if col not in df.columns:
            continue
        roll_mean = df[col].rolling(window, min_periods=60).mean()
        roll_std  = df[col].rolling(window, min_periods=60).std()
        z = (df[col] - roll_mean) / roll_std.replace(0, np.nan)
        df[f"{col}_norm"] = z.clip(-3, 3)
    return df


# ═══════════════════════════════════════════════════════════════════════════════
# FULL PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

def build_features(ticker_csv: str,
                   index_df: pd.DataFrame,
                   horizon: int = 1) -> pd.DataFrame:
    df = pd.read_csv(ticker_csv, index_col="Date", parse_dates=True)

    df = add_return_features(df)
    df = add_momentum_features(df)
    df = add_volatility_features(df)
    df = add_volume_features(df)
    df = add_market_features(df, index_df)
    df = add_target(df, horizon=horizon)
    df = rolling_zscore_normalise(df, FEATURE_COLS)

    # Drop rows where features are NaN (first ~60-252 days of warmup)
    df = df.dropna(subset=[f"{c}_norm" for c in FEATURE_COLS if f"{c}_norm" in df.columns])

    return df


if __name__ == "__main__":
    os.makedirs(FEATURE_DIR, exist_ok=True)

    # Load index once — shared across all stocks
    index_df = pd.read_csv(INDEX_FILE, index_col="Date", parse_dates=True)

    raw_files = [f for f in os.listdir(RAW_DIR)
                 if f.endswith(".csv") and f != "NSEI.csv"]

    for fname in raw_files:
        ticker = fname.replace(".csv", "")
        print(f"Engineering features for {ticker} ...")

        df_feat = build_features(
            ticker_csv=os.path.join(RAW_DIR, fname),
            index_df=index_df,
            horizon=1,     # predict next-day return; change to 5 for weekly
        )

        out_path = os.path.join(FEATURE_DIR, f"{ticker}_features.csv")
        df_feat.to_csv(out_path)
        print(f"  → {len(df_feat)} rows, {len(df_feat.columns)} columns saved to {out_path}")

    print("\nAll features built. Ready for Stage 3 — sequence windowing.")
    print("Feature columns going into LSTM (normalised versions):")
    norm_cols = [f"{c}_norm" for c in FEATURE_COLS]
    for i, col in enumerate(norm_cols, 1):
        print(f"  {i:02d}. {col}")